# Dimensionality Reduction using Autoencoder (MNIST)

In [ ]:
# ============================================================
# Dimensionality Reduction using Autoencoder (MNIST)
# ============================================================

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Load Dataset
# -----------------------------
(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()

print("Training Images :", X_train.shape)
print("Testing Images  :", X_test.shape)

# -----------------------------
# Normalize Images
# -----------------------------
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# -----------------------------
# Flatten Images
# -----------------------------
X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

print("\nFlattened Shape :", X_train.shape)

# -----------------------------
# Latent Dimension
# -----------------------------
latent_dim = 32

# -----------------------------
# Build Encoder
# -----------------------------
encoder = keras.Sequential([

    keras.layers.Input(shape=(784,)),

    keras.layers.Dense(
        256,
        activation="relu"
    ),

    keras.layers.Dense(
        128,
        activation="relu"
    ),

    keras.layers.Dense(
        latent_dim,
        activation="relu"
    )

])

# -----------------------------
# Build Decoder
# -----------------------------
decoder = keras.Sequential([

    keras.layers.Input(shape=(latent_dim,)),

    keras.layers.Dense(
        128,
        activation="relu"
    ),

    keras.layers.Dense(
        256,
        activation="relu"
    ),

    keras.layers.Dense(
        784,
        activation="sigmoid"
    )

])

# -----------------------------
# Build Autoencoder
# -----------------------------
inputs = keras.layers.Input(shape=(784,))

encoded = encoder(inputs)
decoded = decoder(encoded)

autoencoder = keras.Model(
    inputs,
    decoded
)

# -----------------------------
# Model Summary
# -----------------------------
print("\nEncoder Summary")
encoder.summary()

print("\nDecoder Summary")
decoder.summary()

print("\nAutoencoder Summary")
autoencoder.summary()

# -----------------------------
# Compile Model
# -----------------------------
autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

# -----------------------------
# Train Model
# -----------------------------
history = autoencoder.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=1
)

# -----------------------------
# Encode Images
# -----------------------------
compressed_images = encoder.predict(X_test)

print("\nCompressed Representation Shape:")
print(compressed_images.shape)

# -----------------------------
# Decode Images
# -----------------------------
reconstructed_images = decoder.predict(
    compressed_images
)

# -----------------------------
# Plot Training Loss
# -----------------------------
plt.figure(figsize=(8,5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Autoencoder Loss")
plt.legend()

plt.show()

# -----------------------------
# Display Original vs Reconstructed
# -----------------------------
n = 8

plt.figure(figsize=(14,5))

for i in range(n):

    # Original
    plt.subplot(2,n,i+1)
    plt.imshow(
        X_test[i].reshape(28,28),
        cmap="gray"
    )
    plt.title("Original")
    plt.axis("off")

    # Reconstructed
    plt.subplot(2,n,i+n+1)
    plt.imshow(
        reconstructed_images[i].reshape(28,28),
        cmap="gray"
    )
    plt.title("Reconstructed")
    plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# Visualize Latent Features
# -----------------------------
plt.figure(figsize=(10,5))

plt.imshow(
    compressed_images[:20],
    aspect="auto",
    cmap="viridis"
)

plt.colorbar()

plt.xlabel("Latent Features")
plt.ylabel("Images")

plt.title("Compressed Latent Representation")

plt.show()